## Multi-Asset Quantitative Trading & Portfolio Analytics Platform

This project is an interactive multi-asset market research and portfolio analytics platform built with Python and Streamlit. It enables users to search and analyze financial data across equities, ETFs, bonds, commodities, REITs, indexes, and selected crypto proxies; compare assets against benchmarks and optimize portfolios for historical Sharpe ratio; generate BUY / HOLD / SELL signals and backtest systematic allocation strategies.



### Populate the database

**Create market_data_ingest.py**

In [1]:
import os
from datetime import datetime, timedelta

import mysql.connector
import numpy as np
import pandas as pd
import yfinance as yf
from mysql.connector import Error

# ============================================================
# DB CONFIG
# ============================================================
DB_CONFIG = {
    "host": os.getenv("MYSQL_HOST", "localhost"),
    "port": int(os.getenv("MYSQL_PORT", 3306)),
    "user": os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD", "Temara1975@"),
    "database": os.getenv("MYSQL_DATABASE", "Quant_Platform"),
}

# ============================================================
# DEFAULT UNIVERSE
# ============================================================
DEFAULT_UNIVERSE = {
    "Large Cap Stocks": {
        "Apple": "AAPL",
        "Microsoft": "MSFT",
        "NVIDIA": "NVDA",
        "Amazon": "AMZN",
        "Alphabet": "GOOGL",
        "Meta": "META",
        "Berkshire Hathaway": "BRK-B",
        "JPMorgan": "JPM",
        "Johnson & Johnson": "JNJ",
        "Exxon Mobil": "XOM",
        "Visa": "V",
        "Mastercard": "MA",
        "Eli Lilly": "LLY",
        "Costco": "COST",
        "Netflix": "NFLX",
    },
    "Mid Cap Stocks": {
        "DraftKings": "DKNG",
        "Robinhood": "HOOD",
        "RPM International": "RPM",
        "Reliance Steel": "RS",
        "EMCOR": "EME",
        "Duolingo": "DUOL",
        "Cava Group": "CAVA",
    },
    "Small Cap Stocks": {
        "Cleanspark": "CLSK",
        "Astera Labs": "ALAB",
        "TransMedics": "TMDX",
        "Sirius XM": "SIRI",
        "Hims & Hers": "HIMS",
        "SoFi": "SOFI",
    },
    "ETFs": {
        "SPDR S&P 500 ETF": "SPY",
        "Invesco QQQ": "QQQ",
        "iShares Russell 2000 ETF": "IWM",
        "Vanguard Total Stock Market ETF": "VTI",
        "Vanguard FTSE Developed Markets ETF": "VEA",
        "Vanguard FTSE Emerging Markets ETF": "VWO",
        "Financial Select Sector SPDR": "XLF",
        "Technology Select Sector SPDR": "XLK",
        "Energy Select Sector SPDR": "XLE",
    },
    "Bonds": {
        "iShares 20+ Year Treasury Bond ETF": "TLT",
        "iShares 7-10 Year Treasury Bond ETF": "IEF",
        "iShares 1-3 Year Treasury Bond ETF": "SHY",
        "iShares Investment Grade Corporate Bond ETF": "LQD",
        "iShares High Yield Corporate Bond ETF": "HYG",
        "US Aggregate Bond ETF": "AGG",
        "TIPS Bond ETF": "TIP",
    },
    "Commodities": {
        "SPDR Gold Shares": "GLD",
        "iShares Silver Trust": "SLV",
        "United States Oil Fund": "USO",
        "Invesco DB Commodity Index": "DBC",
        "Abrdn Physical Palladium": "PALL",
    },
    "REITs": {
        "Vanguard Real Estate ETF": "VNQ",
        "Realty Income": "O",
        "Simon Property Group": "SPG",
        "Prologis": "PLD",
        "Digital Realty": "DLR",
        "Equinix": "EQIX",
    },
    "Indexes": {
        "S&P 500 Index": "^GSPC",
        "Nasdaq 100 Index": "^NDX",
        "Dow Jones Industrial Average": "^DJI",
        "Russell 2000 Index": "^RUT",
        "VIX": "^VIX",
    },
    "Crypto Proxies": {
        "Bitcoin ETF": "IBIT",
        "Ethereum ETF": "ETHA",
        "Coinbase": "COIN",
        "MicroStrategy": "MSTR",
    },
}

# ============================================================
# DB HELPERS
# ============================================================
def get_connection():
    try:
        return mysql.connector.connect(**DB_CONFIG)
    except Error as e:
        raise RuntimeError(f"Failed to connect to MySQL: {e}") from e


def create_tables():
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS tickers (
                ticker_id INT AUTO_INCREMENT PRIMARY KEY,
                ticker VARCHAR(20) NOT NULL UNIQUE,
                asset_name VARCHAR(255) NOT NULL,
                asset_class VARCHAR(100) NOT NULL,
                sector VARCHAR(100) NULL,
                industry VARCHAR(150) NULL,
                exchange_name VARCHAR(100) NULL,
                currency VARCHAR(20) NULL,
                is_active BOOLEAN NOT NULL DEFAULT TRUE,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
            """
        )

        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS price_history (
                price_id BIGINT AUTO_INCREMENT PRIMARY KEY,
                ticker_id INT NOT NULL,
                trade_date DATE NOT NULL,
                open_price DECIMAL(18,6) NULL,
                high_price DECIMAL(18,6) NULL,
                low_price DECIMAL(18,6) NULL,
                close_price DECIMAL(18,6) NULL,
                adj_close_price DECIMAL(18,6) NULL,
                volume BIGINT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                UNIQUE KEY uq_ticker_date (ticker_id, trade_date),
                CONSTRAINT fk_price_ticker FOREIGN KEY (ticker_id) REFERENCES tickers(ticker_id)
                    ON DELETE CASCADE
            )
            """
        )

        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS signals (
                signal_id BIGINT AUTO_INCREMENT PRIMARY KEY,
                ticker_id INT NOT NULL,
                signal_date DATE NOT NULL,
                signal_type VARCHAR(50) NOT NULL,
                signal_value DECIMAL(18,8) NULL,
                signal_label VARCHAR(20) NOT NULL,
                short_ma INT NULL,
                long_ma INT NULL,
                momentum_window INT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                UNIQUE KEY uq_signal (ticker_id, signal_date, signal_type),
                CONSTRAINT fk_signal_ticker FOREIGN KEY (ticker_id) REFERENCES tickers(ticker_id)
                    ON DELETE CASCADE
            )
            """
        )

        conn.commit()
        print("Tables created successfully.")
    finally:
        cur.close()
        conn.close()


def upsert_ticker(ticker, asset_name, asset_class, sector=None, industry=None, exchange_name=None, currency=None):
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute(
            """
            INSERT INTO tickers (
                ticker, asset_name, asset_class, sector, industry, exchange_name, currency, is_active
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, TRUE)
            ON DUPLICATE KEY UPDATE
                asset_name = VALUES(asset_name),
                asset_class = VALUES(asset_class),
                sector = VALUES(sector),
                industry = VALUES(industry),
                exchange_name = VALUES(exchange_name),
                currency = VALUES(currency),
                is_active = TRUE
            """,
            (ticker, asset_name, asset_class, sector, industry, exchange_name, currency),
        )
        conn.commit()
    finally:
        cur.close()
        conn.close()


def get_ticker_id(ticker):
    conn = get_connection()
    try:
        df = pd.read_sql("SELECT ticker_id FROM tickers WHERE ticker = %s LIMIT 1", conn, params=[ticker])
    finally:
        conn.close()

    if df.empty:
        return None
    return int(df.iloc[0]["ticker_id"])


def upsert_price_history(ticker_id, df_prices: pd.DataFrame):
    if df_prices.empty:
        return

    conn = get_connection()
    cur = conn.cursor()
    try:
        sql = """
            INSERT INTO price_history (
                ticker_id, trade_date, open_price, high_price, low_price, close_price, adj_close_price, volume
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                open_price = VALUES(open_price),
                high_price = VALUES(high_price),
                low_price = VALUES(low_price),
                close_price = VALUES(close_price),
                adj_close_price = VALUES(adj_close_price),
                volume = VALUES(volume)
        """

        records = []
        for idx, row in df_prices.iterrows():
            records.append(
                (
                    ticker_id,
                    pd.to_datetime(idx).date(),
                    safe_float(row.get("Open")),
                    safe_float(row.get("High")),
                    safe_float(row.get("Low")),
                    safe_float(row.get("Close")),
                    safe_float(row.get("Adj Close", row.get("Close"))),
                    safe_int(row.get("Volume")),
                )
            )

        cur.executemany(sql, records)
        conn.commit()
    finally:
        cur.close()
        conn.close()


def upsert_signals(ticker_id, signal_df: pd.DataFrame, signal_type="MA_MOMENTUM"):
    if signal_df.empty:
        return

    conn = get_connection()
    cur = conn.cursor()
    try:
        sql = """
            INSERT INTO signals (
                ticker_id, signal_date, signal_type, signal_value, signal_label, short_ma, long_ma, momentum_window
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                signal_value = VALUES(signal_value),
                signal_label = VALUES(signal_label),
                short_ma = VALUES(short_ma),
                long_ma = VALUES(long_ma),
                momentum_window = VALUES(momentum_window)
        """

        records = []
        for idx, row in signal_df.iterrows():
            records.append(
                (
                    ticker_id,
                    pd.to_datetime(idx).date(),
                    signal_type,
                    safe_float(row.get("signal_value")),
                    row.get("signal_label"),
                    safe_int(row.get("short_ma")),
                    safe_int(row.get("long_ma")),
                    safe_int(row.get("momentum_window")),
                )
            )

        cur.executemany(sql, records)
        conn.commit()
    finally:
        cur.close()
        conn.close()


def safe_float(x):
    if x is None or pd.isna(x):
        return None
    return float(x)


def safe_int(x):
    if x is None or pd.isna(x):
        return None
    return int(x)

In [2]:
# ============================================================
# SIGNAL CALCULATION
# ============================================================
def generate_signal_history(prices: pd.Series, short_ma=50, long_ma=200, momentum_window=60) -> pd.DataFrame:
    df = pd.DataFrame({"Close": prices.copy()}).dropna()
    if df.empty:
        return pd.DataFrame()

    df["short_ma"] = df["Close"].rolling(short_ma).mean()
    df["long_ma"] = df["Close"].rolling(long_ma).mean()
    df["momentum"] = df["Close"].pct_change(momentum_window)

    buy_condition = (df["short_ma"] > df["long_ma"]) & (df["momentum"] > 0)
    sell_condition = (df["short_ma"] < df["long_ma"]) & (df["momentum"] < 0)

    df["signal_label"] = np.where(buy_condition, "BUY", np.where(sell_condition, "SELL", "HOLD"))
    df["signal_value"] = df["momentum"]
    df["short_ma"] = short_ma
    df["long_ma"] = long_ma
    df["momentum_window"] = momentum_window

    return df[["signal_value", "signal_label", "short_ma", "long_ma", "momentum_window"]].dropna(subset=["signal_value"])


# ============================================================
# YFINANCE HELPERS
# ============================================================
def fetch_ticker_metadata(ticker):
    try:
        tk = yf.Ticker(ticker)
        info = tk.info if hasattr(tk, "info") else {}
        return {
            "sector": info.get("sector"),
            "industry": info.get("industry"),
            "exchange_name": info.get("exchange"),
            "currency": info.get("currency"),
            "asset_name": info.get("shortName") or info.get("longName") or ticker,
        }
    except Exception:
        return {
            "sector": None,
            "industry": None,
            "exchange_name": None,
            "currency": None,
            "asset_name": ticker,
        }


def fetch_price_history(ticker, start_date="2020-01-01"):
    try:
        df = yf.download(
            ticker,
            start=start_date,
            auto_adjust=False,
            progress=False,
            threads=False,
        )
        if df is None or df.empty:
            return pd.DataFrame()

        df = df.copy()
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]

        expected_cols = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
        for c in expected_cols:
            if c not in df.columns:
                df[c] = np.nan

        return df[expected_cols].dropna(how="all")
    except Exception as e:
        print(f"Failed downloading {ticker}: {e}")
        return pd.DataFrame()


# ============================================================
# MAIN INGESTION
# ============================================================
def ingest_universe(start_date="2020-01-01"):
    total = 0
    success = 0

    for asset_class, items in DEFAULT_UNIVERSE.items():
        for asset_name, ticker in items.items():
            total += 1
            print(f"\nIngesting {ticker} ...")

            meta = fetch_ticker_metadata(ticker)
            final_name = meta["asset_name"] if meta["asset_name"] and meta["asset_name"] != ticker else asset_name

            try:
                upsert_ticker(
                    ticker=ticker,
                    asset_name=final_name,
                    asset_class=asset_class,
                    sector=meta["sector"],
                    industry=meta["industry"],
                    exchange_name=meta["exchange_name"],
                    currency=meta["currency"],
                )

                ticker_id = get_ticker_id(ticker)
                if ticker_id is None:
                    print(f"Could not get ticker_id for {ticker}")
                    continue

                prices = fetch_price_history(ticker, start_date=start_date)
                if prices.empty:
                    print(f"No price data for {ticker}")
                    continue

                upsert_price_history(ticker_id, prices)

                close_series = prices["Adj Close"].fillna(prices["Close"]).dropna()
                signal_df = generate_signal_history(close_series, short_ma=50, long_ma=200, momentum_window=60)
                if not signal_df.empty:
                    upsert_signals(ticker_id, signal_df, signal_type="MA_MOMENTUM")

                success += 1
                print(f"Done: {ticker}")

            except Exception as e:
                print(f"Error ingesting {ticker}: {e}")

    print("\n==================================================")
    print(f"Ingestion finished. Success: {success}/{total}")
    print("==================================================")


if __name__ == "__main__":
    create_tables()
    ingest_universe(start_date="2020-01-01")

Tables created successfully.

Ingesting AAPL ...


/var/folders/c4/xntd0yb16dz1d4dv_zrhdtvc0000gn/T/ipykernel_56354/3649874266.py:217: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT ticker_id FROM tickers WHERE ticker = %s LIMIT 1", conn, params=[ticker])


Done: AAPL

Ingesting MSFT ...


/var/folders/c4/xntd0yb16dz1d4dv_zrhdtvc0000gn/T/ipykernel_56354/3649874266.py:217: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT ticker_id FROM tickers WHERE ticker = %s LIMIT 1", conn, params=[ticker])


Done: MSFT

Ingesting NVDA ...


...

Done: ETHA

Ingesting COIN ...


/var/folders/c4/xntd0yb16dz1d4dv_zrhdtvc0000gn/T/ipykernel_56354/3649874266.py:217: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT ticker_id FROM tickers WHERE ticker = %s LIMIT 1", conn, params=[ticker])


Done: COIN

Ingesting MSTR ...


/var/folders/c4/xntd0yb16dz1d4dv_zrhdtvc0000gn/T/ipykernel_56354/3649874266.py:217: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT ticker_id FROM tickers WHERE ticker = %s LIMIT 1", conn, params=[ticker])


Done: MSTR

Ingestion finished. Success: 64/64
